In [78]:
# EJERCICIO INTEGRADOR
# Vas a construir un sistema simple de análisis de ventas
# que lea datos, los procese y genere reportes.

# Los datos van a venir de un archivo CSV que primero debes crear.
# El sistema debe:

In [79]:
# 1. DATOS — crea un archivo "datos.csv" con estos campos:
#    vendedor, mes, monto, region, producto
#    con al menos 10 registros inventados por ti

ventas = [
    {'vendedor': 'Ohiggins', 'mes': 'junio', 'monto': 100000, 'region': 'sur', 'producto': 'procesador'},
    {'vendedor': 'Jorge', 'mes': 'enero', 'monto': 70000, 'region': 'centro', 'producto': 'xbox'},
    {'vendedor': 'Ivan', 'mes': 'abril', 'monto': 90000, 'region': 'centro', 'producto': 'macbook'},
    {'vendedor': 'Mario', 'mes': 'mayo', 'monto': 200000, 'region': 'norte', 'producto': 'ropa'},
    {'vendedor': 'Cesar', 'mes': 'noviembre', 'monto': 500000, 'region': 'centro', 'producto': 'escritorio'},
    {'vendedor': 'Dulce', 'mes': 'diciembre', 'monto': 70000, 'region': 'norte', 'producto': 'pulparindo'},
    {'vendedor': 'Emmanuel', 'mes': 'febrero', 'monto': 90000, 'region': 'centro', 'producto': 'chilaquil'},
    {'vendedor': 'Sam', 'mes': 'agosto', 'monto': 20000, 'region': 'norte', 'producto': 'nerf'},
    {'vendedor': 'Vero', 'mes': 'septiembre', 'monto': 40000, 'region': 'centro', 'producto': 'comida'},
    {'vendedor': 'Andrea', 'mes': 'julio', 'monto': 50000, 'region': 'sur', 'producto': 'iphone'}
]

import csv
from pathlib import Path

PATH_BASE = Path.cwd()
PATH_CSV = PATH_BASE / 'csv' / 'datos.csv'
PATH_JSON = PATH_BASE / 'json' / 'reporte.json'


with open(PATH_CSV, "w", newline="") as archivo:
    campos = ['vendedor', 'mes',  'monto', 'region', 'producto']
    writer = csv.DictWriter(archivo, fieldnames=campos)

    writer.writeheader()
    writer.writerows(ventas)

In [80]:
# 2. CLASE VentaInvalidaError — error personalizado para validaciones

class VentaInvalidaError(Exception):
    pass

In [81]:
# 3. CLASE Venta — representa una venta individual con:
#    - __init__ que reciba vendedor, mes, monto, region, producto
#    - validación: monto debe ser positivo, si no lanza VentaInvalidaError
#    - __str__ que muestre todos los campos

class Venta:
    def __init__(self, vendedor, mes, monto, region, producto):
        if monto < 0:
            raise VentaInvalidaError('El monto debe ser positivo')
        self.vendedor = vendedor
        self.mes = mes
        self.monto = monto
        self.region = region
        self.producto = producto

    def __str__(self):
        return f"Vendedor: {self.vendedor}, Mes: {self.mes}, Monto: {self.monto}, Region: {self.region}, Producto: {self.producto}"

In [84]:
# 4. CLASE AnalizadorVentas con:
#    - __init__ que reciba la ruta del CSV
#    - método cargar() que lea el CSV y cree objetos Venta
#      guardándolos en una lista self.ventas
#    - método total_por_vendedor() que retorne un dict con el total de cada vendedor
#    - método top_ventas(n) que retorne las n ventas más altas usando sorted y lambda
#    - método reporte() que guarde un JSON con:
#      * total por vendedor
#      * la venta más alta
#      * promedio general de todas las ventas

class AnalizadorVentas:
    def __init__(self, PATH_CSV, PATH_JSON):
        self.PATH_CSV = PATH_CSV
        self.PATH_JSON = PATH_JSON
        self.ventas = []

    def cargar(self):
        import csv
        with open(self.PATH_CSV, 'r') as archivo:
            reader = csv.DictReader(archivo)
            lineas_leidas = []
            for linea in reader:
                linea['monto'] = int(linea['monto'])
                lineas_leidas.append(linea)

                venta_obj = Venta(linea['vendedor'], linea['mes'], linea['monto'], linea['region'], linea['producto'])
                self.ventas.append(venta_obj)
            
        return self.ventas

    def total_por_vendedor(self):
        total = {}
        for v in self.ventas:
            nombre = v.vendedor
            if nombre not in total:
                total[nombre] = 0

            total[nombre] += v.monto
        
        return total
    
    def top_ventas(self, n):
        top = sorted(self.ventas, key=lambda v: v.monto, reverse=True)
        top = top[: n]
        for t in top:
            print(t)

    def reporte(self):
        import json

        venta_mayor = max(self.ventas, key=lambda v: v.monto)

        reporte = {
            'total_por_vendedor': self.total_por_vendedor(),
            'venta_mas_alta': {
                'vendedor': venta_mayor.vendedor,
                'monto': venta_mayor.monto,
                'producto': venta_mayor.producto
            },
            'promedio_general': sum(v.monto for v in self.ventas) / len(self.ventas)
        }

        with open(self.PATH_JSON, 'w') as archivo:
            json.dump(reporte, archivo, indent=4)

In [85]:
# 5. USA el AnalizadorVentas para:
#    - cargar los datos
#    - imprimir el total por vendedor
#    - imprimir el top 3 ventas
#    - generar el reporte JSON

analizador = AnalizadorVentas(PATH_CSV, PATH_JSON)
analizador.cargar()
analizador.total_por_vendedor()
analizador.top_ventas(5)
analizador.reporte()

Vendedor: Cesar, Mes: noviembre, Monto: 500000, Region: centro, Producto: escritorio
Vendedor: Mario, Mes: mayo, Monto: 200000, Region: norte, Producto: ropa
Vendedor: Ohiggins, Mes: junio, Monto: 100000, Region: sur, Producto: procesador
Vendedor: Ivan, Mes: abril, Monto: 90000, Region: centro, Producto: macbook
Vendedor: Emmanuel, Mes: febrero, Monto: 90000, Region: centro, Producto: chilaquil
